In [20]:
from envs.factory import make_custom_env
from envs.ant_maze import sample_random_goal, all_possible_goals
from wrappers import (
    LogWrapper,
    BraxGymnaxWrapper,
    VecEnv,
    NormalizeVecObservation,
    NormalizeVecReward,
    ClipAction,
)
import jax
import jax.numpy as jnp
from ppo_continuous_action_custom_brax import ActorCritic

In [2]:
custom_env = make_custom_env(
        env_name="ant_u_maze",
        backend=None,
        env_kwargs={},
    )
env = BraxGymnaxWrapper(
        env=custom_env,
        episode_length=1000,
        action_repeat=1,
    )
env_params = None
env = LogWrapper(env)
env = ClipAction(env)
env = VecEnv(env)

network = ActorCritic(
        env.action_space(env_params).shape[0], activation="relu", hidden_dim=16
    )

rng = jax.random.PRNGKey(0)
rng, _rng = jax.random.split(rng)
init_x = jnp.zeros(env.observation_space(env_params).shape)
goal_dim = 2
init_x = jnp.concatenate([init_x, jnp.zeros((goal_dim, ))], axis=-1)
network_params = network.init(_rng, init_x)

In [16]:
import jax
import jax.numpy as jnp

def evaluate_multiple_goals(env, brax_env, network, params, goals, num_envs_per_goal, max_steps=1000):
    """Evaluate success rate for each goal over multiple random starts.

    Uses the full wrapped env stack (VecEnv + LogWrapper + ...). VecEnv.reset
    already vmaps over envs, so each goal is evaluated with a batched reset of
    ``num_envs_per_goal`` keys rather than vmapping over scalar keys.
    """
    env_params = None

    def _reinit_with_goal(brax_state, goal):
        q = brax_state.pipeline_state.q.at[-2:].set(goal)
        pipeline_state = brax_env.pipeline_init(q, brax_state.pipeline_state.qd)
        obs = brax_env._get_obs(pipeline_state)
        return brax_state.replace(pipeline_state=pipeline_state, obs=obs)

    reinit_with_goal = jax.vmap(_reinit_with_goal, in_axes=(0, None))

    def eval_one_goal(rng, specific_goal):
        reset_rngs = jax.random.split(rng, num_envs_per_goal)
        obsv, env_state = env.reset(reset_rngs, env_params)

        brax_state = reinit_with_goal(env_state.env_state, specific_goal)
        env_state = env_state.replace(env_state=brax_state)
        obsv = brax_state.obs

        goal_batch = jnp.broadcast_to(
            specific_goal, (num_envs_per_goal, specific_goal.shape[-1])
        )

        def step_fn(carry, _):
            obsv, env_state, rng = carry
            rng, step_rng, action_rng = jax.random.split(rng, 3)
            step_rngs = jax.random.split(step_rng, num_envs_per_goal)
            policy_obs = jnp.concatenate([obsv, goal_batch], axis=-1)
            pi, _ = network.apply(params, policy_obs)
            action = pi.sample(seed=action_rng)
            obsv, env_state, reward, done, info = env.step(
                step_rngs, env_state, action, env_params
            )
            success = env_state.env_state.metrics["success"]
            return (obsv, env_state, rng), success

        _, successes = jax.lax.scan(step_fn, (obsv, env_state, rng), None, length=max_steps)
        
        return successes.max(axis=0).mean()

    vmap_goals = jax.vmap(eval_one_goal, in_axes=(0, 0))
    goal_rngs = jax.random.split(jax.random.PRNGKey(42), goals.shape[0])
    return vmap_goals(goal_rngs, goals)

In [ ]:
# goals = jnp.array([
#     [4.0, 4.0],
#     [4.0, 6.0],
#     [4.0, 7.0],
#     [4.0, 8.0],
#     [4.0, 9.0],

# ])
all_goals = all_possible_goals()
evaluate_multiple_goals(env, custom_env, network, network_params, goals, 4)

[[ 4  8]
 [ 4 12]
 [ 8 12]
 [12  4]
 [12  8]
 [12 12]]
